# Introduction 

This notebook is for analyzing MS2 spectra.

# Terminology

- **MS1 scan**: A common full range scan that measures the m/z of all ions in the sample. Also called a "survey scan".
- **MS2 scan**: A scan that measures the m/z of fragments of a specific ion. Also called a "MS/MS scan".
- **Parent peak**: The m/z of the ion that was selected for fragmentation in an MS2 scan. Also called the "precursor peak".
- **HCD energy**: The energy used to fragment the parent peak in an MS2 scan. HCD stands for "higher-energy collisional dissociation". It's measured in volts (V) and is instrument-specific.
- **Isolation window**: The m/z range around the parent peak that is isolated for fragmentation in an MS2 scan.

# Setup

In [ ]:
from utils import CompositionMap, DataExtractor, Ms2Dashboard

from mascope_sdk import MascopeClient


mascope = MascopeClient()
# List samples in a batch
samples = mascope.samples.list(
    dataset="My Dataset", batch="My Batch"
)  # <- replace with your batch name

samples[["sample_item_id", "sample_item_name", "filename"]]

# Data Extraction

The MS1 and MS2 spectra are loaded as centroids. By default, only the centroids with the signal-to-noise ratio (SNR) above 10 are retained. You can adjust this threshold by changing the `noise_threshold` parameter in the `peak_filtering_params`.

Another important parameter is `parent_peak_tolerance`, which controls how closely parent peaks from different MS2 scans must match to be considered the same. This is necessary because of instrument float drifts, usually within 0.001 Da. By default, peaks within 0.001 Da of each other are collapsed into a single representative m/z (the median of the cluster, rounded to 4 decimal places).

Please make sure the number of detected parent peaks matches your expectation.

In [ ]:
SAMPLE_ITEM_ID = "sample-123"  # Replace with your actual sample item ID

peak_filtering_params = {"noise_threshold": 10, "parent_peak_tolerance": 0.001}
data = DataExtractor(mascope, SAMPLE_ITEM_ID, peak_filtering_params)
print(f"Clustered parent peaks: {data.parent_peaks}")

# Composition Assignment

In the next cell, you can assign elemental composition to the peaks. The parameters for the assignment can be set in the `params` dictionary.

Since the calculation of the corresponding elemental composition is purely mathematical (based on the closest mass and `element_count_ranges`), some of the results may not be chemically valid. To filter out such results, heuristic rules and isotopic pattern matching are applied. The heuristic rules can be customized by passing a dictionary with the parameters to the `apply_heuristic_rules` function.

In [ ]:
from mascope_tools.composition import (
    CompositionSearchConfig,
    HeuristicFilterConfig,
)


search_config = CompositionSearchConfig(
    element_count_ranges="C0-50 H0-100 O0-50 [15N]0-1",
    use_unsaturation=True,
    min_unsaturation=-1000,
    max_unsaturation=1000,
    ionizations="-",
    only_integer_unsaturation=False,
    peak_height_threshold=0,
    mass_range_ppm=10,
)
heuristic_config = HeuristicFilterConfig(
    carbon_element_ratio_range={
    "H/C": (0.1, 6.0),
    "N/C": (0.0, 2.0),
    "O/C": (0.0, 2.0),
    "S/C": (0.0, 0.1),
    "P/C": (0.0, 0.05),
    "Cl/C": (0.0, 0.05),
    "Br/C": (0.0, 0.05),
    "F/C": (0.0, 0.05),
    "I/C": (0.0, 0.05),
},
    non_carbon_element_ratio_range={},
)

compositions = CompositionMap(data, search_config, heuristic_config)

# Analysis

Here you can visually inspect the fragments and their assigned compositions.

In [ ]:
dashboard = Ms2Dashboard(data, compositions)
dashboard.show_fragments()

Timeseries of fragments can be plotted for diagnostics. To do that, uncomment the code in the cell below. By default, the timeseries of the top 3 most intense fragments are plotted. You can change this number by adjusting the `n_fragments` parameter. You can also choose to normalize the intensities by the total ion current (TIC) per MS2 scan by setting `normalize_by="tic"`.

Since timeseries calculation is computationally intensive, timeseries for all fragments are not calculated by default. They are requested from the server on demand when you choose a parent peak. This means, timeseries are not shown instantly when you select a parent peak. Please allow some time for the timeseries to load. 

In [ ]:
# dashboard.show_timeseries(normalize_by=None, n_fragments=3)

# Fragment Table

In the table below, each row corresponds to a fragment detected in the MS2 scan. Ten rows are shown by default, the number of rows is controlled by `max_fragments` argument. Of course, some of these fragments may be noise or contaminants.

The parent peak metadata is shown at the top of the table, derived from the MS1 scan survey. 

To download the fragment table as a CSV, click the `Export CSV` button. The data will be packed and the download button will appear in the bottom of the table.

In [ ]:
from utils.fragment_table import FragmentTable


table = FragmentTable(data, compositions, max_fragments=10)
table.show()

# Ratio Chart (Declustering)

Although `RatioChart` is called "declustering", it does not know about declustering as a mechanism. It only plots “target fragment versus parent” for the passed `target_fragment` string. Its fragment intensity stays at zero unless there is an exact formula match. If that exact ion label is missing or spelled differently from your `target_fragment`, the blue fraction is 0%, and you see only grey. Formulae used:

$$\text{fragment fraction} = \frac{I_{fr}}{I_{fr} + I_p} * 100$$
$$\text{parent fraction} = \frac{I_p}{I_{fr} + I_p} * 100$$

where $I_p$ and $I_{fr}$  are intensities of parent and fragment.

In [ ]:
from utils.ratio_chart import RatioChart

# Filter parent peaks to a specific m/z range for visualization
mz_min = 0
mz_max = 1000

parent_filter = {
    float(pp) for pp in data.parent_peaks
    if mz_min <= float(pp) <= mz_max
}

chart = RatioChart(
    data,
    compositions,
    target_fragment="NO3-",
    parent_filter=parent_filter,
)
chart.show()

# Fragmentation Analysis

In opposite to `RatioChart`, `FragmentationAnalysis` can show blue in desclustering spectra, because it is looking for the reagent ion by m/z rather than relying on that exact label. Formulae:

$$\text{parent fraction} = \frac{I_p}{I_{re} + I_p} * 100$$
$$\text{reagent fraction} = \frac{I_{re}}{I_{re} + I_p} * 100$$

where $I_p$ and $I_{re}$ are intensities of parent and reagent ion.

In [ ]:
from utils import FragmentationAnalysis

analysis = FragmentationAnalysis(data, compositions, reagent="+[15N]O3-")
analysis.show()
analysis.classification